## RDFの生成
- 個別に作成する必要はない

In [10]:
import numpy as np
import multiprocessing
import codecs
import tqdm
import trimesh

from tqdm import tqdm
from rdflib import Graph, URIRef, BNode, Literal, plugin
from rdflib.namespace import RDF, RDFS, OWL,Namespace, NamespaceManager
from rdflib.serializer import Serializer
import ifcopenshell
from ifcopenshell import geom

In [11]:
# ここを任意のパスに変えてください
ifc_path = './files/ToyodaLab.ifc'
ifc_path = './files/TKY-080c_INTEC_Interior_A22.ifc'
ifc_path = './files/TKY-080_Exterior model_A22.ifc'
rdf_path = "./output/{}.ttl".format(ifc_path.split('/')[-1].split('.')[0])

# IfcOpenShellの設定
try:
    ifc_file = ifcopenshell.open(ifc_path)
except:
    print(ifcopenshell.get_log())
else:
    # https://blenderbim.org/docs-python/ifcopenshell/geometry_settings.html
    settings = geom.settings()
    

In [12]:
# 任意に設定する
site_name = ifc_path.split('/')[-1].split('.')[0].replace(' ', '_')
host = 'http://takenaka.co.jp/futaba/{0}/'.format(site_name)

device = host + 'device#'
building = host + 'building#'
floor = host + 'storey#'
space = host + 'space#'

bot = 'https://w3id.org/bot#'
rdfs_a = 'http://www.w3.org/2000/01/rdf-schema#type'
iot = 'http://iotschema.org/'
mozzila = 'https://iot.mozilla.org/schemas/'
brick = 'https://brickschema.org/ontology/1.3'

# 注：ifcOWLではないことに注意．以下の名前空間は独自定義
ifc = 'http://www.buildingsmart-tech.org/IFC2x3#'


host_ns = Namespace(host)
device_ns = Namespace(device)
building_ns = Namespace(building)
floor_ns = Namespace(floor)
space_ns = Namespace(space)
# recも使える
# https://dev.realestatecore.io/ontology/
bot_ns = Namespace(bot)
ifc_ns = Namespace(ifc)

namespace_manager = NamespaceManager(Graph())
namespace_manager.bind('inst', host_ns, override=False)
namespace_manager.bind('bot', bot_ns, override=False)
namespace_manager.bind('ifc', ifc_ns, override=False)
namespace_manager.bind('device', device_ns, override=False)
namespace_manager.bind('building', building_ns, override=False)
namespace_manager.bind('storey', floor_ns, override=False)
namespace_manager.bind('space', space_ns, override=False)


In [13]:
def format_ifc_info(info) -> dict:
    """ IfcOpenShellの情報はmultiprocessingを含んでいるので一般オブジェクトに変換する
    
    Args:
        info (_type_): IfcEntityのメタデータ

    Returns:
        dict: _description_
    """
    ret = dict()
    for key in info.keys():
        item = info.get(key)
        if type(item) != ifcopenshell.entity_instance and item != None:
            ret[key] = item
    return ret


def get_project_info(ifc_file, settings, name='Sample') :
    """ プロジェクトの情報を取得する

    Args:
        ifc_file (_type_): _description_
        settings (_type_): _description_
        name (str, optional): _description_. Defaults to 'Sample'.

    Returns:
        str, str, str: _description_
    """
    import math
    # IfcProjectから名前をとる
    prj = ifc_file.by_type('IfcProject')[0]
    name_ = prj.LongName if prj.LongName != 'プロジェクト名' else name
    name_ = name if name_ is None else name_
    
    # IfcSiteから緯度経度を取得
    site = ifc_file.by_type('IfcSite')[0]
    lat = '.'.join([str(i) for i in site.RefLatitude]) if site.RefLatitude is not None else ""
    lon = '.'.join([str(i) for i in site.RefLongitude]) if site.RefLongitude is not None else ""
 
    return name_, lat, lon


def get_properties(element):
    """ IFCのオブジェクトよりプロパティを抽出

    Args:
        element (_type_): _description_

    Returns:
        dict : プロパティセット
    """
    ret = vars(element)

    # propertyがない場合はエラーになる
    if hasattr(element, 'IsDefinedBy'):
        for tmp in element.IsDefinedBy:
            if tmp.is_a('IfcRelDefinesByProperties'):            
                pset = tmp.RelatingPropertyDefinition
                if pset.is_a('IfcPropertySet'):
                    for prop in pset.HasProperties:
                        try:
                            name = prop.Name
                            val = prop.NominalValue
                            ret[name] = val.wrappedValue
                        except:
                            print('Invalid Property: ' + name)
                elif pset.is_a('IfcElementQuantity'):
                    # 面積の場合がある
                    name = pset.Name
                    measurement = pset.MethodOfMeasurement 
                    quantities = pset.Quantities[0] 
                    #from IPython.core.debugger import Pdb; Pdb().set_trace()
                    if quantities.is_a('IfcQuantityArea'):
                        label = quantities.Name.replace(' ', '_')
                        description = quantities.Description
                        unit = quantities.Unit
                        # 記法の変更
                        area_value = quantities.AreaValue
                        ret[label]  = area_value
                else:
                    print('Invalid Type: ' + vars(pset)['type'])
            elif tmp.is_a('IfcRelDefinesByType'):
                #TODO クラス（Family）の定義
                pass
            else:
                print('Invalid Type : ' + vars(tmp)['type'])
      
    # 不要なプロパティの刈込(主観で決めている)
    ret.pop('OwnerHistory', None)
    ret.pop('CompositionType', None)
    ret.pop('Representation', None)
    ret.pop('ObjectPlacement', None)
    # ObjectTypeとほぼ等価
    ret.pop('Reference', None)
    # その他
    addr = ret.pop('BuildingAddress', None)
    if addr:
        addr = addr.AddressLines[0]
        ret['Address'] = addr
    return ret


def create_resource(g, entity, class_, name_=None, ns_=None, ADD_ID=True):
    
    props = get_properties(entity)
    if 'Name' in props and props['Name'] is not None:
        name =  get_unicode(props['Name']) 
    else:
        name = props['type'] 
    ns = ns_ if ns_ is not None else host_ns
    inst_name = "_".join([name.split(":")[0].replace(" ", "_"), str(props['id'])]) if ADD_ID else name.split(":")[0]
    inst_name =  inst_name if name_ is None else name_
    resource = ns.term(inst_name)
    add_literal(g, resource, props)
    g.add((resource, RDF.type, class_))
    return resource


def add_literal(g, resource, props):
    for label in props:
        if props[label]:
            name = get_unicode(label).strip().replace(' ', '').replace('　', '')
            value = get_unicode(props[label])
            #TODO ここを適当なクラスに置き換えると素敵
            g.add((resource, ifc_ns.term(name), Literal(value)))


def get_unicode(val):
    return codecs.decode(val, 'unicode-escape') if '\\u' in str(val) else val

In [14]:

def get_OBJ_format(vertices, uvs, normals, faceVertIDs, uvIDs, normalIDs, vertexColors, name):
    ret = []
    ret.append("####\n")
    ret.append("#\n")
    ret.append("# Vertices: %s\n" %(len(vertices)))
    ret.append("# Faces: %s\n" %(len( faceVertIDs)))
    ret.append("#\n")
    ret.append("####\n")
    ret.append("g %s\n" % name)
    for vi, v in enumerate( vertices ):
        vStr = "v %s %s %s"  %(v[0], v[1], v[2])
        if len( vertexColors) > 0:
            color = vertexColors[vi]
            vStr += " %s %s %s" %(color[0], color[1], color[2])
        vStr += "\n"
        ret.append(vStr)
    for uv in uvs:
        uvStr =  "vt %s %s\n"  %(uv[0], uv[1])
        ret.append(uvStr)
    for n in normals:
        nStr =  "vn %s %s %s\n"  %(n[0], n[1], n[2])
        ret.append(nStr)
    for fi, fvID in enumerate( faceVertIDs ):
        fStr = "f"
        for fvi, fvIDi in enumerate( fvID ):
            fStr += " %s" %( fvIDi)
            if len(uvIDs) > 0:
                fStr += "/%s" %( uvIDs[fi][fvi] )
            if len(normalIDs) > 0:
                fStr += "/%s" %( normalIDs[fi][fvi] )
        fStr += "\n"
        ret.append(fStr)
    return ''.join(ret)

def get_vertices(entity, z):
    shape = geom.create_shape(settings, entity)
    # メッシュの取得
    ios_vertices = shape.geometry.verts
    ios_faces = shape.geometry.faces 
    # 3つずつに変形する
    ret_vertices = []
    ret_faces = []
    for i in range(0, len(ios_vertices), 3):
        ret_vertices.append([ios_vertices[i], ios_vertices[i+1], ios_vertices[i+2]+z])        
    for i in range(0, len(ios_faces), 3):
        ret_faces.append([ios_faces[i]+1, ios_faces[i+1]+1, ios_faces[i+2]+1])
    return ret_vertices, ret_faces

def get_vertices2(entity, z):
    shape = geom.create_shape(settings, entity)
    # メッシュの取得
    ios_vertices = shape.geometry.verts
    ios_faces = shape.geometry.faces 
    # 3つずつに変形する
    ret_vertices = []
    ret_faces = []
    for i in range(0, len(ios_vertices), 3):
        ret_vertices.append([ios_vertices[i], ios_vertices[i+1], ios_vertices[i+2]+z])        
    for i in range(0, len(ios_faces), 3):
        ret_faces.append([ios_faces[i], ios_faces[i+1], ios_faces[i+2]])
    return ret_vertices, ret_faces

def extract_geometry(space, elevation, v_count):
    # SpaceにはIfcExtrudedAreaSolidが定義されている。
    # ソリッドモデルやBREPSの扱いは面倒なので、ポリゴンで出力する。 => BOT:hasSimple3DModel
    vertices, faceVertIDs  = get_vertices(space, elevation)
    faceVertIDs = [(np.array(faceVertID) + v_count).tolist() for faceVertID in faceVertIDs]
    obj_str = get_OBJ_format(vertices, [], [], faceVertIDs, [], [], [], space.GlobalId)
    v_count = v_count + len(vertices)
    return obj_str, v_count

def extract_geometry2(space, elevation):
    # SpaceにはIfcExtrudedAreaSolidが定義されている。
    # ソリッドモデルやBREPSの扱いは面倒なので、ポリゴンで出力する。 => BOT:hasSimple3DModel
    vertices, faceVertIDs  = get_vertices2(space, elevation)
    return vertices, faceVertIDs

In [15]:
g = Graph()
g.namespace_manager = namespace_manager

# SITEは1つと仮定する。名前を書き換える
entity = ifc_file.by_type("IfcSite")[0]
site_resource = create_resource(g, entity, bot_ns.Site, site_name)

# Building
entity = entity.IsDecomposedBy[0].RelatedObjects[0]
building_name = None
building_resource = create_resource(g, entity, bot_ns.Building, building_name, building_ns)

# リンク生成
g.add((site_resource, bot_ns.hasBuilding, building_resource))

In [16]:
# TODO 構成システム類の抽出（空調等のシステムを表すIfcSystem)
systems = entity.ServicedBySystems
print(len(systems))

storeys = entity.IsDecomposedBy[0].RelatedObjects
storeys


0


(#121=IfcBuildingStorey('3DIZOtvqHBYedwsDw_SBB5',#12,'1FL',$,$,#119,$,$,.ELEMENT.,0.),
 #149536=IfcBuildingStorey('0dDIdpwPz0xuAPU_Zv9MW7',#12,'2FL',$,$,#149535,$,$,.ELEMENT.,4600.),
 #305536=IfcBuildingStorey('0ip8CrmHv8NAGMh_luvn7B',#12,'3FL',$,$,#305535,$,$,.ELEMENT.,9000.),
 #725183=IfcBuildingStorey('0duX_XvfXES910TXsvBGja',#12,'4FL',$,$,#725182,$,$,.ELEMENT.,13400.),
 #726890=IfcBuildingStorey('1GwSVedYL1jxZEk8AzFqR5',#12,'5FL',$,$,#726889,$,$,.ELEMENT.,17800.),
 #728996=IfcBuildingStorey('2ibvGxBBbBxRJTPY1b2wMR',#12,'6FL',$,$,#728995,$,$,.ELEMENT.,22200.),
 #731088=IfcBuildingStorey('34R7ohIILBYxPrPbqOhY8Q',#12,'7FL',$,$,#731087,$,$,.ELEMENT.,26575.),
 #816941=IfcBuildingStorey('0kptvu91b4OwVVvdWihSTC',#12,'PHFL',$,$,#816940,$,$,.ELEMENT.,30625.))

In [17]:

scene = trimesh.Scene()

for storey in tqdm(storeys):
    storey_pset = get_properties(storey)      
    elevation = storey.Elevation
    # TODO 語彙の修正　（FL -> F)
    if 'Name' in storey_pset and storey_pset['Name'] is not None:
        storey_name = storey_pset['Name'].replace(' ', '_')
    else:
        storey_name = 'NoName_Floor'
    
    storey_resource = create_resource(g, storey, bot_ns.Storey, storey_name, floor_ns, ADD_ID=False)           
    g.add((building_resource, bot_ns.hasStorey, storey_resource))
    tmp = []
    print(storey_name)
    space_list = []
    
    # ちゃんとIfcSpaceが定義されていればここに来る
    if len(storey.IsDecomposedBy) > 0:
        spaces = storey.IsDecomposedBy[0].RelatedObjects
        for space in spaces:
            space_resource = create_resource(g, space, bot_ns.Space, None, space_ns)
            g.add((storey_resource, bot_ns.hasSpace, space_resource))

            # BuildingElementProxyとか、IfcFlowTerminalなどの空間などに含まれる要素
            # 壁とかは取れない。要考察。
            contains_elements = space.ContainsElements
            for e in contains_elements:
                # IfcRelContainedInSpatialStructure
                for element in e.RelatedElements:
                    elem_resource = create_resource(g, element, bot_ns.Element, None, device_ns)
                    g.add((space_resource, bot_ns.containsElement, elem_resource))
                    tmp.append(element.GlobalId)
                    
            if space.Representation:
                vertices, faceVertIDs = extract_geometry2(space, elevation/1000)
                #TODO 階層構造化
                mesh = trimesh.Trimesh(vertices=vertices, faces=faceVertIDs)
                pset = get_properties(space)   
                # node_nameは識別性を意識してLongName+GUIDとする。
                node_name = scene.add_geometry(mesh,geom_name=pset['GlobalId'], node_name=get_unicode(pset['LongName']) + '_' + str(pset['id']))
                # print(' ' + node_name)
                space_list.append(node_name)
                
    # TODO ここには壁とか什器とか窓とかシステムとか
    contains_elements = storey.ContainsElements

    for e in contains_elements:
        for elem in e.RelatedElements: 
            if elem.GlobalId in tmp:
                print("Ignore : " + elem.Name)
                pass
            else:
                # 多分こういうのは他にもある
                if elem.is_a("IfcGrid"):
                    continue
                elem_resource = create_resource(g, elem, bot_ns.Element)
                g.add((storey_resource, bot_ns.containsElement, elem_resource))

 25%|██▌       | 2/8 [00:00<00:00, 14.57it/s]

1FL
2FL
3FL


 88%|████████▊ | 7/8 [00:00<00:00, 29.44it/s]

4FL
5FL
6FL
7FL


100%|██████████| 8/8 [00:00<00:00, 29.34it/s]

PHFL


In [18]:
# 保存
with open(rdf_path, 'w', encoding='utf-8') as f:
    f.write(g.serialize(format="turtle").decode())

In [19]:
scene.show()

ValueError: Can't export empty scenes!

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt


# 空のNetworkxグラフを作成
nx_graph = nx.Graph()

# RDFデータをNetworkxグラフに変換
for subject, predicate, object in g:
    nx_graph.add_edge(subject, object)
    
# Networkxグラフの可視化
nx.draw(nx_graph, with_labels=True)

# グラフを表示
plt.show()